# NTUC Income Eco Run: results export

Pulls every result from the public RaceRoster API and writes the CSVs that Power BI imports.

- Raw API responses are cached in `raw/`. A request is skipped when its file already exists, so a re-run makes no API calls.
- CSVs are written to `output/`.
- Run all cells from top to bottom. The completeness check stops the run if any race is missing participants.

In [1]:
import itertools
import json
import math
import threading
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd
import requests

BASE_URL = "https://results.raceroster.com/v2/api/"
REQUEST_HEADERS = {"Accept": "application/json", "User-Agent": "Mozilla/5.0"}
MAX_CONCURRENT_REQUESTS = 4
# The API sits behind an AWS firewall that challenges ~1,000 requests/minute; this keeps us near 200/minute.
MIN_SECONDS_BETWEEN_REQUESTS = 0.3
WAF_COOLDOWN_SECONDS = 300
MAX_ATTEMPTS = 5
RETRYABLE_STATUS_CODES = {429, 500, 502, 503, 504}

RAW_DIR = Path("raw")
OUTPUT_DIR = Path("output")

YEARS = {
    2024: "p2m7kjgqjqkzjrwn",
    2025: "ru6ha7aauyfgsk4b",
    2026: "agmtjs3rt7d9e5bb",
}

# Race ID -> clean race name. Races are matched by ID, never by the API's name or distance.
RACES = {
    # 2024
    199895: "21.1km", 199896: "15km", 199897: "10km",
    199898: "5km",    199899: "3km",  200770: "700m Kids",
    # 2025
    230117: "21.1km", 230113: "15km", 230114: "10km",
    230115: "5km",    230116: "3km",  230742: "700m Kids",
    # 2026
    255209: "21.1km", 255210: "15km", 255211: "10km",
    255212: "5km",    255213: "3km",  255215: "700m Kids",
}

YEARS_TO_RUN = list(YEARS)

**What this does:** loads the libraries and sets the fixed inputs: API address, request headers, the concurrency limit, the three event codes, and the table mapping each race ID to a clean race name.

**Why:** race names and the API's `distance` field don't match across years, so races are identified only by ID. Adding 2027 means one new line in `YEARS` and six in `RACES`. `YEARS_TO_RUN` controls which years this run fetches.

In [2]:
network_call_count = 0
network_call_lock = threading.Lock()


def record_network_call():
    global network_call_count
    with network_call_lock:
        network_call_count += 1


request_pacing_lock = threading.Lock()
next_request_time = 0.0


def wait_for_request_slot():
    """Space requests at least MIN_SECONDS_BETWEEN_REQUESTS apart across all threads."""
    global next_request_time
    with request_pacing_lock:
        now = time.monotonic()
        wait_seconds = next_request_time - now
        next_request_time = max(now, next_request_time) + MIN_SECONDS_BETWEEN_REQUESTS
    if wait_seconds > 0:
        time.sleep(wait_seconds)


def fetch_text(url, params=None):
    """GET the URL, retrying on 429, 5xx, network errors and firewall rate-limit challenges."""
    for attempt in range(1, MAX_ATTEMPTS + 1):
        is_last_attempt = attempt == MAX_ATTEMPTS
        wait_for_request_slot()
        record_network_call()
        try:
            response = requests.get(url, params=params, headers=REQUEST_HEADERS, timeout=30)
        except (requests.ConnectionError, requests.Timeout):
            if is_last_attempt:
                raise
            time.sleep(2 ** attempt)
            continue

        # The firewall answers HTTP 202 with an empty body and this header when requests come too fast.
        if "x-amzn-waf-action" in response.headers:
            if is_last_attempt:
                raise RuntimeError("Still rate-limited by the API firewall. Re-run later; the cache keeps all progress.")
            print(f"  rate-limited by the API firewall; waiting {WAF_COOLDOWN_SECONDS // 60} minutes")
            time.sleep(WAF_COOLDOWN_SECONDS)
            continue

        if response.status_code in RETRYABLE_STATUS_CODES and not is_last_attempt:
            time.sleep(2 ** attempt)
            continue

        response.raise_for_status()
        return response.text


def load_or_fetch(cache_path, url, params=None):
    """Return the parsed JSON from cache_path, fetching and saving it first if the file doesn't exist."""
    if cache_path.exists():
        return json.loads(cache_path.read_text(encoding="utf-8"))

    body = fetch_text(url, params)
    parsed = json.loads(body)
    if parsed.get("data") is None:
        # Never cache a response without data: it would be reused on every future run.
        raise ValueError(f"Response from {url} (params {params}) has no 'data': {body[:200]}")

    cache_path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = cache_path.with_suffix(".tmp")
    temporary_path.write_text(body, encoding="utf-8")
    # Rename last, so a crash mid-write never leaves a half-written file that looks cached.
    temporary_path.replace(cache_path)
    return parsed


def run_in_parallel(function, items, progress_every=None):
    """Apply function to every item using up to MAX_CONCURRENT_REQUESTS threads, keeping input order."""
    results = []
    with ThreadPoolExecutor(max_workers=MAX_CONCURRENT_REQUESTS) as executor:
        for completed_count, result in enumerate(executor.map(function, items), start=1):
            results.append(result)
            if progress_every is not None and completed_count % progress_every == 0:
                print(f"  {completed_count:,} / {len(items):,}")
    return results


def event_url(year):
    return BASE_URL + "events/" + YEARS[year]


def make_name_key(name):
    """Lowercase, trim, collapse spaces and sort the words, so "Lim Eugene" and "Eugene Lim" share a key.

    Word order is ignored because 2025 lists most names first-name-first while 2024 and 2026 mostly
    list them surname-first. A missing name gives an empty key.
    """
    if name is None:
        return ""
    words = name.lower().split()
    return " ".join(sorted(words))

**What this does:** defines the helpers the rest of the notebook uses. `load_or_fetch` reads a response from `raw/` if it is already there; otherwise it calls the API and saves the response. Requests are spaced about 0.3 s apart with up to 4 at once. If the API's firewall pushes back (HTTP 202 with an `x-amzn-waf-action` header), the helper waits 5 minutes before retrying.

**Why:** the API sits behind an AWS firewall that blocked this notebook at about 1,000 requests a minute, so slow and steady finishes sooner than fast and blocked. Caching every response means a crash partway through resumes rather than restarting, and a re-run makes zero API calls (`network_call_count` proves it). Responses without data are never cached, because a bad cached response would be reused on every future run. The live tracker had exactly this bug with empty results.

In [3]:
sub_events_by_year = {}
for year in YEARS_TO_RUN:
    cache_path = RAW_DIR / str(year) / "event.json"
    response = load_or_fetch(cache_path, event_url(year))
    sub_events_by_year[year] = response["data"]["event"]["subEvents"]

for year, sub_events in sub_events_by_year.items():
    print(year)
    for sub_event in sub_events:
        if sub_event["id"] in RACES:
            scope_label = RACES[sub_event["id"]]
        else:
            scope_label = "(out of scope)"
        print(f"  {sub_event['id']}  {sub_event['name']:<25} {sub_event['resultCount']:>6,}  -> {scope_label}")

2024
  199895  21.1km                     1,629  -> 21.1km
  199896  15km                         488  -> 15km
  199897  10km                       1,280  -> 10km
  199898  5km                          788  -> 5km
  199899  3km                          235  -> 3km
  200769  1KM (E-cert only)             77  -> (out of scope)
  200770  700M (E-cert only)           184  -> 700m Kids
2025
  230117  21.1KM Half Marathon       1,952  -> 21.1km
  230113  15KM                         483  -> 15km
  230114  10KM                       1,406  -> 10km
  230115  5KM                          842  -> 5km
  230116  3KM                          219  -> 3km
  230741  Kids - 1KM                    67  -> (out of scope)
  230742  Kids - 700m                  201  -> 700m Kids
  230743  Pets - 1KM                   152  -> (out of scope)
2026
  255209  21.1KM Half Marathon       2,471  -> 21.1km
  255210  15KM                         500  -> 15km
  255211  10KM                       1,707  -> 10km
  25521

**What this does:** fetches each year's race list, with each race's ID, name and official `resultCount`, and shows how every race maps to the `RACES` table.

**Why:** `resultCount` is the official total the completeness check compares against. The printout makes it easy to confirm that the six in-scope races per year are correct and that everything else is out of scope.

In [4]:
# The largest race has about 2,500 runners; the completeness check stops the run if a listing is ever cut short.
LISTING_LIMIT = 5000


def load_race_listing(year, sub_event):
    """Every participant in one race, from the same listing the RaceRoster results page shows."""
    cache_path = RAW_DIR / str(year) / "listing" / f"{sub_event['id']}.json"
    url = f"{BASE_URL}result-events/{sub_event['resultEventId']}/sub-events/{sub_event['id']}/results"
    return load_or_fetch(cache_path, url, params={"start": 0, "limit": LISTING_LIMIT})


# (year, race_id) -> results RaceRoster counts but does not list publicly
hidden_count_by_race = {}


def discover_participants(year):
    participants_by_id = {}
    for sub_event in sub_events_by_year[year]:
        listing = load_race_listing(year, sub_event)
        hidden_count_by_race[(year, sub_event["id"])] = listing["meta"]["totalResults"] - listing["meta"]["filteredResults"]
        for row in listing["data"]:
            if row["id"] in participants_by_id:
                continue
            participants_by_id[row["id"]] = {
                "year": year,
                "race_id": sub_event["id"],
                "race_name": sub_event["name"],
                "participant_id": row["id"],
                "name": row["name"],
                "name_key": make_name_key(row["name"]),
                "bib": row["bib"],
            }

    print(f"{year}: {len(participants_by_id):,} unique participants across {len(sub_events_by_year[year])} races")
    return list(participants_by_id.values())


participant_records = []
for year in YEARS_TO_RUN:
    participant_records.extend(discover_participants(year))

all_participants = pd.DataFrame(participant_records)

2024: 4,671 unique participants across 7 races
2025: 5,322 unique participants across 8 races
2026: 6,626 unique participants across 9 races


**What this does:** for every race in every year, including kids' 1km and pets races, fetches the full list of participants in one call. This is the same list the RaceRoster results page shows. Each participant is kept once, keyed by their API `id`.

**Why:** the spec's `participant-search` endpoint turned out to be fuzzy and capped at 100 results. Even after narrowing capped searches, it never returned 2024's 5-digit 3km bibs (0 of 235 found). The race listing returns everyone in a race directly. It was checked against the search results for 2026 21.1km: identical 2,471 IDs, names and bibs. Dedupe by `id`, never by name, because two different people can share a name in the same race.

In [5]:
completeness_rows = []
for year in YEARS_TO_RUN:
    participants_this_year = all_participants[all_participants["year"] == year]
    for sub_event in sub_events_by_year[year]:
        race_id = sub_event["id"]
        if race_id not in RACES:
            continue
        found_count = int((participants_this_year["race_id"] == race_id).sum())
        completeness_rows.append({
            "year": year,
            "race": RACES[race_id],
            "race_id": race_id,
            "result_count": sub_event["resultCount"],
            "found": found_count,
            "gap": sub_event["resultCount"] - found_count,
            "hidden_by_raceroster": hidden_count_by_race[(year, race_id)],
        })

completeness = pd.DataFrame(completeness_rows)
print(completeness.to_string(index=False))

unexplained_gaps = completeness[completeness["gap"] != completeness["hidden_by_raceroster"]]
if not unexplained_gaps.empty:
    raise RuntimeError(
        "STOP: found count does not match resultCount, and the gap is not explained by results RaceRoster hides. "
        "Report this before continuing; do not fetch details."
    )

hidden_total = int(completeness["hidden_by_raceroster"].sum())
if hidden_total > 0:
    print(f"\nAll in-scope races complete except {hidden_total} result(s) RaceRoster counts but does not publish.")
else:
    print("\nAll in-scope races complete.")

 year      race  race_id  result_count  found  gap  hidden_by_raceroster
 2024    21.1km   199895          1629   1629    0                     0
 2024      15km   199896           488    487    1                     1
 2024      10km   199897          1280   1280    0                     0
 2024       5km   199898           788    788    0                     0
 2024       3km   199899           235    235    0                     0
 2024 700m Kids   200770           184    184    0                     0
 2025    21.1km   230117          1952   1952    0                     0
 2025      15km   230113           483    483    0                     0
 2025      10km   230114          1406   1406    0                     0
 2025       5km   230115           842    842    0                     0
 2025       3km   230116           219    219    0                     0
 2025 700m Kids   230742           201    201    0                     0
 2026    21.1km   255209          2471   2471    0 

**What this does:** for each in-scope race, compares the number of unique participants found with the official `resultCount`, and halts the notebook if any race is short. The only allowed gap is one that RaceRoster itself reports as hidden: counted in `totalResults` but not published in the listing (`filteredResults`).

**Why:** a missing participant would make every later number silently wrong. RaceRoster withholds a small number of results from public view (for example, one 2024 15km result), and no public endpoint returns them. The table shows those separately so the gap is documented rather than hidden.

In [6]:
def load_detail(year, participant_id):
    cache_path = RAW_DIR / str(year) / "detail" / f"{participant_id}.json"
    url = event_url(year) + "/detail/" + participant_id
    response = load_or_fetch(cache_path, url)
    return response["data"]["result"]


def whole_seconds(chip_time_sec):
    """A few 2024 times have tenths (2647.6). RaceRoster's displayed chip time rounds these up, so this does too."""
    if chip_time_sec is None:
        return None
    return math.ceil(chip_time_sec)


in_scope_participants = all_participants[all_participants["race_id"].isin(RACES.keys())]

result_records = []
for year in YEARS_TO_RUN:
    participants_this_year = in_scope_participants[in_scope_participants["year"] == year]
    participant_ids = list(participants_this_year["participant_id"])
    print(f"{year}: loading {len(participant_ids):,} details")
    details = run_in_parallel(lambda participant_id: load_detail(year, participant_id), participant_ids, progress_every=500)

    for participant, detail in zip(participants_this_year.itertuples(), details):
        detail_race_id = int(detail["resultSubEventId"])
        if detail_race_id != participant.race_id:
            raise RuntimeError(
                f"Race mismatch for {participant.participant_id}: search says {participant.race_id}, detail says {detail_race_id}"
            )
        result_records.append({
            "year": year,
            "race": RACES[participant.race_id],
            "race_id": participant.race_id,
            "participant_id": participant.participant_id,
            "name": participant.name,
            "name_key": participant.name_key,
            "bib": participant.bib,
            "chip_time_sec": whole_seconds(detail["chipTimeSec"]),
            "chip_time": detail["chipTime"],
            "race_status": detail["raceStatus"],
        })

results = pd.DataFrame(result_records)
results["chip_time_sec"] = results["chip_time_sec"].astype("Int64")
print(f"\n{len(results):,} result rows. Network calls so far this run: {network_call_count:,}")

2024: loading 4,603 details


  500 / 4,603


  1,000 / 4,603


  1,500 / 4,603


  2,000 / 4,603


...

  4,500 / 5,103
  5,000 / 5,103
2026: loading 6,181 details


  500 / 6,181
  1,000 / 6,181
  1,500 / 6,181


  2,000 / 6,181
  2,500 / 6,181
  3,000 / 6,181


  3,500 / 6,181
  4,000 / 6,181
  4,500 / 6,181
  5,000 / 6,181


  5,500 / 6,181
  6,000 / 6,181

15,887 result rows. Network calls so far this run: 0


**What this does:** fetches the detail record (chip time and race status) for every participant in an in-scope race, 10 at a time, caching each one in `raw/{year}/detail/`. It also checks that the race in the detail record matches the race from the listing, after converting both IDs to integers. Fourteen 2024 times come back with tenths of a second (`2647.6`); these are rounded up, which reproduces RaceRoster's displayed `chip_time` in every case.

**Why:** chip times are only available from the per-participant detail endpoint, which means about 12,000 calls in total. The cache lets a crash resume where it stopped, and the race cross-check catches any mismatch between the two endpoints before it reaches the CSV.

In [7]:
OUTPUT_DIR.mkdir(exist_ok=True)

results = results.sort_values(["year", "race_id", "chip_time_sec", "participant_id"], na_position="last")
results.to_csv(OUTPUT_DIR / "results.csv", index=False)

all_participants_columns = ["year", "race_id", "race_name", "participant_id", "name", "name_key"]
all_participants = all_participants.sort_values(["year", "race_id", "participant_id"])
all_participants[all_participants_columns].to_csv(OUTPUT_DIR / "all_participants.csv", index=False)

print(f"results.csv:          {len(results):,} rows")
print(f"all_participants.csv: {len(all_participants):,} rows")

results.csv:          15,887 rows
all_participants.csv: 16,619 rows


**What this does:** writes `results.csv` (one row per participant per year, in-scope races only, all statuses kept) and `all_participants.csv` (everyone from every race).

**Why:** these are the files Power BI imports. `chip_time_sec` is a nullable whole-number column, so a missing time is left blank rather than written as `0` or `NaN`. `bib` stays as text.

In [8]:
def unique_name_keys(year):
    """name_keys that appear exactly once in this year, counted across all races including out-of-scope ones."""
    name_key_counts = all_participants.loc[all_participants["year"] == year, "name_key"].value_counts()
    unique_keys = set(name_key_counts[name_key_counts == 1].index)
    # Participants with no name can't be told apart, so they are never linked.
    unique_keys.discard("")
    return unique_keys


def has_valid_time(result_row):
    if result_row["race_status"] != "COMPLETE":
        return False
    if pd.isna(result_row["chip_time_sec"]):
        return False
    return result_row["chip_time_sec"] > 0


repeat_runner_columns = [
    "name_key", "year_from", "year_to", "race_from", "race_to", "same_race",
    "time_from_sec", "time_to_sec", "delta_sec",
]
repeat_runner_records = []

for year_from, year_to in itertools.combinations(sorted(YEARS_TO_RUN), 2):
    linkable_keys = unique_name_keys(year_from) & unique_name_keys(year_to)

    rows_from = results[(results["year"] == year_from) & results["name_key"].isin(linkable_keys)].set_index("name_key")
    rows_to = results[(results["year"] == year_to) & results["name_key"].isin(linkable_keys)].set_index("name_key")
    # A name whose single entry is in an out-of-scope race (e.g. a pets race) has no row in results, so it drops out here.
    shared_keys = sorted(set(rows_from.index) & set(rows_to.index))

    for name_key in shared_keys:
        row_from = rows_from.loc[name_key]
        row_to = rows_to.loc[name_key]
        same_race = row_from["race"] == row_to["race"]

        if same_race and has_valid_time(row_from) and has_valid_time(row_to):
            time_from_sec = row_from["chip_time_sec"]
            time_to_sec = row_to["chip_time_sec"]
            delta_sec = time_to_sec - time_from_sec
        else:
            time_from_sec = None
            time_to_sec = None
            delta_sec = None

        repeat_runner_records.append({
            "name_key": name_key,
            "year_from": year_from,
            "year_to": year_to,
            "race_from": row_from["race"],
            "race_to": row_to["race"],
            "same_race": same_race,
            "time_from_sec": time_from_sec,
            "time_to_sec": time_to_sec,
            "delta_sec": delta_sec,
        })

repeat_runners = pd.DataFrame(repeat_runner_records, columns=repeat_runner_columns)
for time_column in ["time_from_sec", "time_to_sec", "delta_sec"]:
    repeat_runners[time_column] = repeat_runners[time_column].astype("Int64")
repeat_runners.to_csv(OUTPUT_DIR / "repeat_runners.csv", index=False)

print(f"repeat_runners.csv:   {len(repeat_runners):,} rows")

repeat_runners.csv:   2,796 rows


**What this does:** links the same `name_key` between two years only if it appears exactly once in each year, counting every race in that year, including out-of-scope ones. The check is done separately for each pair of years. A time change (`delta_sec`, negative = faster) is filled in only when both years are the same race and both rows are `COMPLETE` with a time above 0.

**Why:** there is no participant ID that carries across years, so the name is the only link. `name_key` ignores word order, because 2025 writes most names first-name-first ("Eugene Lim") while 2024 and 2026 mostly write them surname-first ("Kwong Matthew"); keeping word order linked only 68 runners from 2024 to 2025 instead of 863. Ignoring order also makes "Tan Eric" and "Eric Tan" in the same year count as a duplicate, so neither links. The rules avoid false merges at the cost of missing some real repeat runners. A few participants have no name at all (group bibs in 2024); they stay in `results.csv` but are never linked. Known limitation: two different people who each have a unique name in their year can still be merged, and this can't be fixed with the data available.

In [9]:
race_order = list(dict.fromkeys(RACES.values()))

print("Rows per race x year (should match resultCount above)")
rows_per_race = pd.crosstab(results["race"], results["year"]).reindex(race_order)
print(rows_per_race.to_string())

print("\nrace_status per year")
status_counts = results.groupby("year")["race_status"].value_counts(dropna=False)
print(status_counts.to_string())
non_complete_count = int((results["race_status"] != "COMPLETE").sum())
if non_complete_count > 0:
    print(f"FLAG: {non_complete_count:,} rows are not COMPLETE. A finishers-vs-non-finishers view may be possible.")
else:
    print("All rows are COMPLETE.")

print("\nRows with no name (kept in results, never linked as repeat runners)")
unnamed_counts = results[results["name_key"] == ""].groupby(["year", "race"]).size()
if unnamed_counts.empty:
    print("  none")
else:
    print(unnamed_counts.to_string())

if 2024 in YEARS_TO_RUN:
    kids_2024 = results[(results["year"] == 2024) & (results["race"] == "700m Kids")]
    timed_kids_count = int((kids_2024["chip_time_sec"].fillna(0) > 0).sum())
    print(f"\n700m Kids 2024: {timed_kids_count} of {len(kids_2024)} rows have chip_time_sec > 0")
    if timed_kids_count < len(kids_2024) / 2:
        print("FLAG: most 2024 700m Kids rows have no time (race was 'E-cert only'). Kept in the CSV; decide in Power BI.")

print("\nRaces where every runner has the same time (placeholder, not real timing)")
timed_results = results[results["chip_time_sec"].fillna(0) > 0]
distinct_times = timed_results.groupby(["year", "race"])["chip_time_sec"].nunique()
placeholder_races = distinct_times[distinct_times == 1]
if placeholder_races.empty:
    print("  none")
else:
    for (year, race), _ in placeholder_races.items():
        placeholder_seconds = int(timed_results[(timed_results["year"] == year) & (timed_results["race"] == race)]["chip_time_sec"].iloc[0])
        print(f"  FLAG: {year} {race}: every time is {placeholder_seconds} s. Exclude from time analysis in Power BI.")

print("\nRepeat runners per year pair")
if repeat_runners.empty:
    print("  none (needs at least two years in YEARS_TO_RUN)")
else:
    repeat_summary = repeat_runners.groupby(["year_from", "year_to"]).agg(
        linked=("name_key", "size"),
        same_race=("same_race", "sum"),
        same_race_with_delta=("delta_sec", "count"),
    )
    print(repeat_summary.to_string())

Rows per race x year (should match resultCount above)
year       2024  2025  2026
race                       
21.1km     1629  1952  2471
15km        487   483   500
10km       1280  1406  1707
5km         788   842   964
3km         235   219   246
700m Kids   184   201   293

race_status per year
year  race_status
2024  COMPLETE       4601
      IN_PROGRESS       2
2025  COMPLETE       5103
2026  COMPLETE       6181
FLAG: 2 rows are not COMPLETE. A finishers-vs-non-finishers view may be possible.

Rows with no name (kept in results, never linked as repeat runners)
year  race
2024  10km    26
      5km      3

700m Kids 2024: 184 of 184 rows have chip_time_sec > 0

Races where every runner has the same time (placeholder, not real timing)
  FLAG: 2024 700m Kids: every time is 3600 s. Exclude from time analysis in Power BI.
  FLAG: 2025 700m Kids: every time is 420 s. Exclude from time analysis in Power BI.
  FLAG: 2026 700m Kids: every time is 600 s. Exclude from time analysis in Power

**What this does:** prints the end-of-run summary: rows per race and year, `race_status` counts (flagged if anything other than `COMPLETE` appears), how many 2024 700m Kids rows have a time, any race where every runner has the identical time, and how many runners were linked for each pair of years.

**Why:** these are the numbers to check before building anything in Power BI. The flags point out anything that could change what analysis is possible, such as non-finishers, or a kids' race whose 'times' are one placeholder value (all 184 of the 2024 700m Kids rows have a time, but every one is 1:00:00).

In [10]:
duplicate_pair_count = int(results.duplicated(["year", "participant_id"]).sum())
assert duplicate_pair_count == 0, f"{duplicate_pair_count} duplicate (year, participant_id) pairs"
print("OK  no duplicate (year, participant_id) pairs")

if 2026 in YEARS_TO_RUN:
    repeat_rows_touching_2026 = repeat_runners[(repeat_runners["year_from"] == 2026) | (repeat_runners["year_to"] == 2026)]

    tan_sean_key = make_name_key("Tan Sean")
    tan_sean_2026 = results[(results["year"] == 2026) & (results["name_key"] == tan_sean_key)]
    assert {"2548", "1251"} <= set(tan_sean_2026["bib"]), "both 2026 Tan Seans should be in results"
    assert tan_sean_key not in set(repeat_rows_touching_2026["name_key"]), "tan sean must not link into 2026"
    print(f"OK  both 2026 Tan Seans in results (bibs {sorted(tan_sean_2026['bib'])}), neither linked")

    eric_tan_key = make_name_key("Eric Tan")
    eric_tan_2026 = all_participants[(all_participants["year"] == 2026) & (all_participants["name_key"] == eric_tan_key)]
    assert eric_tan_key not in set(repeat_rows_touching_2026["name_key"]), "eric tan must not link into or out of 2026"
    print(f"OK  eric tan not linked to 2026 (appears in 2026 races: {sorted(eric_tan_2026['race_name'])})")

print(f"\nNetwork calls this run: {network_call_count:,} (0 means everything came from the cache)")

OK  no duplicate (year, participant_id) pairs
OK  both 2026 Tan Seans in results (bibs ['1251', '2548']), neither linked
OK  eric tan not linked to 2026 (appears in 2026 races: ['10KM', '21.1KM Half Marathon', '700m - Pets'])

Network calls this run: 0 (0 means everything came from the cache)


**What this does:** runs the acceptance checks from the spec: no duplicate participant per year, both 2026 "Tan Sean"s present and not linked, and "eric tan" not linked into or out of 2026. It then reports how many API calls this run made.

**Why:** these checks confirm that the matching rules behaved as intended on known tricky cases. On a second run, a network call count of 0 proves everything came from the cache.